# Predicting Residential EV Charging Loads using Neural Networks

Predict EV charging loads (kWh) from Norwegian residential charging data using traffic features and a PyTorch neural network.

**Pipeline:** Load EV charging reports and local traffic data → merge on plug-in hour → clean and encode → 80/20 train/test split → linear regression baseline → PyTorch MLP. Target: `El_kWh`.

In [81]:
# Setup
import numpy as np
import pandas as pd

## 1. Load and Merge Datasets

We combine EV charging sessions with hourly traffic counts from nearby locations. The hypothesis: traffic density may correlate with charging behavior (e.g., more cars on the road when people plug in).

**EV charging reports** — Norwegian residential buildings. Key fields: plug-in/plug-out times, `El_kWh` (target), duration, month, weekday.

In [82]:
ev_charging_reports = pd.read_csv(
    "datasets/EV charging reports.csv",
    sep=';',           # delimiter is semicolon, not comma
    header=0,
    na_values=['NA']    # treat "NA" as missing (used in Shared_ID)
)
ev_charging_reports.head()

,session_ID,Garage_ID,User_ID,User_type,Shared_ID,Start_plugin,Start_plugin_hour,End_plugout,End_plugout_hour,El_kWh,Duration_hours,month_plugin,weekdays_plugin,Plugin_category,Duration_category
0,1,AdO3,AdO3-4,Private,NaN,21.12.2018 10:20,10,21.12.2018 10:23,10.0,"0,3","0,05",Dec,Friday,late morning (9-12),Less than 3 hours
1,2,AdO3,AdO3-4,Private,NaN,21.12.2018 10:24,10,21.12.2018 10:32,10.0,"0,87","0,136666667",Dec,Friday,late morning (9-12),Less than 3 hours
2,3,AdO3,AdO3-4,Private,NaN,21.12.2018 11:33,11,21.12.2018 19:46,19.0,"29,87","8,216388889",Dec,Friday,late morning (9-12),Between 6 and 9 hours
3,4,AdO3,AdO3-2,Private,NaN,22.12.2018 16:15,16,23.12.2018 16:40,16.0,"15,56","24,41972222",Dec,Saturday,late afternoon (15-18),More than 18 hours
4,5,AdO3,AdO3-2,Private,NaN,24.12.2018 22:03,22,24.12.2018 23:02,23.0,"3,62","0,970555556",Dec,Monday,late evening (21-midnight),Less than 3 hours


*Columns: session_ID, Garage_ID, User_ID, User_type, Shared_ID, Start_plugin, Start_plugin_hour, End_plugout, End_plugout_hour, El_kWh (target), Duration_hours, month_plugin, weekdays_plugin, Plugin_category, Duration_category*

In [83]:
ev_charging_reports.iloc[0].value_counts

<bound method IndexOpsMixin.value_counts of session_ID                             1
Garage_ID                           AdO3
User_ID                           AdO3-4
User_type                        Private
Shared_ID                            NaN
Start_plugin            21.12.2018 10:20
Start_plugin_hour                     10
End_plugout             21.12.2018 10:23
End_plugout_hour                    10.0
El_kWh                               0,3
Duration_hours                      0,05
month_plugin                         Dec
weekdays_plugin                   Friday
Plugin_category      late morning (9-12)
Duration_category      Less than 3 hours
Name: 0, dtype: object>

**Traffic data** — Hourly vehicle counts at 5 locations. Joined to charging sessions by plug-in hour.

In [84]:
traffic_reports = pd.read_csv('datasets/Local traffic distribution.csv', sep=';', header=0)
traffic_reports.head()

,Date_from,Date_to,KROPPAN BRU,MOHOLTLIA,SELSBAKK,MOHOLT RAMPE 2,Jonsvannsveien vest for Steinanvegen
0,01.12.2018 00:00,01.12.2018 01:00,639,0,0,4,144
1,01.12.2018 01:00,01.12.2018 02:00,487,153,115,21,83
2,01.12.2018 02:00,01.12.2018 03:00,408,85,75,10,69
3,01.12.2018 03:00,01.12.2018 04:00,282,89,56,8,39
4,01.12.2018 04:00,01.12.2018 05:00,165,64,34,3,25


*Columns: Date_from, Date_to, plus 5 traffic location counts (KROPPAN BRU, MOHOLTLIA, SELSBAKK, MOHOLT RAMPE 2, Jonsvannsveien vest for Steinanvegen)*


In [85]:
traffic_reports.iloc[0].value_counts

<bound method IndexOpsMixin.value_counts of Date_from                               01.12.2018 00:00
Date_to                                 01.12.2018 01:00
KROPPAN BRU                                          639
MOHOLTLIA                                              0
SELSBAKK                                               0
MOHOLT RAMPE 2                                         4
Jonsvannsveien vest for Steinanvegen                 144
Name: 0, dtype: object>

Merge on `Start_plugin_hour` (ev) and `Date_from` (traffic). `Start_plugin_hour` is normalized to `HH:00` format for the join.

In [86]:
# This takes the string, drops the last 2 characters (the minutes), and adds '00'
ev_charging_reports['Start_plugin_hour'] = ev_charging_reports['Start_plugin'].str[:-2] + '00'

ev_charging_traffic = pd.merge(ev_charging_reports, traffic_reports, left_on='Start_plugin_hour', right_on='Date_from')

In [87]:
ev_charging_traffic.info()

<class 'pandas.DataFrame'>
RangeIndex: 6878 entries, 0 to 6877
Data columns (total 22 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   session_ID                            6878 non-null   int64  
 1   Garage_ID                             6878 non-null   str    
 2   User_ID                               6878 non-null   str    
 3   User_type                             6878 non-null   str    
 4   Shared_ID                             1412 non-null   str    
 5   Start_plugin                          6878 non-null   str    
 6   Start_plugin_hour                     6878 non-null   str    
 7   End_plugout                           6844 non-null   str    
 8   End_plugout_hour                      6844 non-null   float64
 9   El_kWh                                6878 non-null   str    
 10  Duration_hours                        6844 non-null   str    
 11  month_plugin                

## 2. Data Cleaning and Preparation

Remove non-predictive columns (IDs, timestamps), fix European decimal format, and encode categoricals for the neural network. Rows with missing target or key features are dropped.

Drop IDs, timestamps, and categorical labels. Keep: `User_type`, `El_kWh`, `Duration_hours`, month/weekday (for encoding), and traffic counts.

In [88]:
ev_charging_traffic = ev_charging_traffic.drop(['session_ID', 'Garage_ID', 'User_ID', 
                'Shared_ID',
                'Plugin_category','Duration_category', 
                'Start_plugin', 'Start_plugin_hour', 'End_plugout', 'End_plugout_hour', 
                'Date_from', 'Date_to'], axis=1)

ev_charging_traffic.info()

<class 'pandas.DataFrame'>
RangeIndex: 6878 entries, 0 to 6877
Data columns (total 10 columns):
 #   Column                                Non-Null Count  Dtype
---  ------                                --------------  -----
 0   User_type                             6878 non-null   str  
 1   El_kWh                                6878 non-null   str  
 2   Duration_hours                        6844 non-null   str  
 3   month_plugin                          6878 non-null   str  
 4   weekdays_plugin                       6878 non-null   str  
 5   KROPPAN BRU                           6878 non-null   str  
 6   MOHOLTLIA                             6878 non-null   str  
 7   SELSBAKK                              6878 non-null   str  
 8   MOHOLT RAMPE 2                        6878 non-null   int64
 9   Jonsvannsveien vest for Steinanvegen  6878 non-null   int64
dtypes: int64(2), str(8)
memory usage: 537.5 KB


Replace `,` with `.` in `El_kWh` and `Duration_hours`; coerce traffic columns to numeric.

In [89]:
ev_charging_traffic['El_kWh'] = ev_charging_traffic['El_kWh'].str.replace(',', '.')
ev_charging_traffic['Duration_hours'] = ev_charging_traffic['Duration_hours'].str.replace(',', '.')

# Convert traffic columns to numeric (they may load as strings)
traffic_cols = ['KROPPAN BRU', 'MOHOLTLIA', 'SELSBAKK', 'MOHOLT RAMPE 2', 'Jonsvannsveien vest for Steinanvegen']
for col in traffic_cols:
    ev_charging_traffic[col] = pd.to_numeric(ev_charging_traffic[col], errors='coerce')

# print first 5 rows
ev_charging_traffic.head()

,User_type,El_kWh,Duration_hours,month_plugin,weekdays_plugin,KROPPAN BRU,MOHOLTLIA,SELSBAKK,MOHOLT RAMPE 2,Jonsvannsveien vest for Steinanvegen
0,Private,0.3,0.05,Dec,Friday,3244.0,1632.0,545.0,194,622
1,Private,0.87,0.136666667,Dec,Friday,3244.0,1632.0,545.0,194,622
2,Private,29.87,8.216388889,Dec,Friday,3605.0,1691.0,605.0,230,771
3,Private,15.56,24.41972222,Dec,Saturday,3052.0,1484.0,453.0,224,694
4,Private,3.62,0.970555556,Dec,Monday,1390.0,693.0,226.0,83,353


`User_type` → binary (Private=1). Month and weekday → one-hot. Drop rows with missing target; fill remaining NaNs with 0.

In [90]:
# Encode categorical columns for neural network
ev_charging_traffic['User_type'] = (ev_charging_traffic['User_type'] == 'Private').astype(float)
ev_charging_traffic = pd.get_dummies(ev_charging_traffic, columns=['month_plugin', 'weekdays_plugin'], drop_first=False)

# Drop rows with missing target or key features
ev_charging_traffic = ev_charging_traffic.dropna(subset=['El_kWh', 'Duration_hours'])

# Fill any remaining NaN (e.g. in traffic columns) with 0
ev_charging_traffic = ev_charging_traffic.fillna(0)

# Convert remaining columns to float
ev_charging_traffic = ev_charging_traffic.astype(float)

## 3. Train/Test Split

80% train, 20% test. Features `X` exclude `El_kWh`; target `y` is `El_kWh`. Fixed `random_state` for reproducibility.

In [91]:
X = ev_charging_traffic.drop('El_kWh', axis=1)
y = ev_charging_traffic['El_kWh']

In [92]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=2)

## 4. Linear Regression Baseline

Linear regression gives a simple baseline. If the neural network doesn't beat it, the extra complexity isn't justified. MSE (and √MSE ≈ average error in kWh) is the metric.

In [93]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()

lr.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [94]:
from sklearn.metrics import mean_squared_error

X_predictions = lr.predict(X_test)

test_mse = mean_squared_error(X_predictions, y_test)

test_mse

120.88389109081311

## 5. Neural Network (PyTorch)

MLP with two hidden layers. MSE loss, Adam optimizer. The nonlinearity can capture interactions (e.g., traffic × time of day) that linear regression misses.

In [95]:
import torch
from torch import nn
from torch import optim

In [96]:
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float).reshape(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float).reshape(-1, 1)

**Architecture:** input → 56 (ReLU) → 26 (ReLU) → 1. Input size matches feature count after one-hot encoding.

In [97]:
torch.manual_seed(42)

# Use actual number of features (varies with one-hot encoding of month/weekday)
n_features = X_train.shape[1]
model = nn.Sequential(
    nn.Linear(n_features, 56),
    nn.ReLU(),
    nn.Linear(56, 26),
    nn.ReLU(),
    nn.Linear(26, 1)
)

In [98]:
loss = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0007)

In [99]:
max_epochs = 4500

for epoch in range(max_epochs):
    # 1. forward pass
    y_pred = model(X_train_tensor)
    
    # 2. compute loss
    train_loss = loss(y_pred, y_train_tensor)
    
    # 3. zero gradients
    optimizer.zero_grad()
    
    # 4. backward pass
    train_loss.backward()
    
    # 5. optimizer step
    optimizer.step()
    
    # 6. print progress
    if epoch % 500 == 0:
        print(f'Epoch [{epoch}/{max_epochs}], Loss: {train_loss.item():.4f}')

Epoch [0/4500], Loss: 4520.4883
Epoch [500/4500], Loss: 153.3723
Epoch [1000/4500], Loss: 143.0740
Epoch [1500/4500], Loss: 137.5213
Epoch [2000/4500], Loss: 132.2448
Epoch [2500/4500], Loss: 125.8603
Epoch [3000/4500], Loss: 121.2663
Epoch [3500/4500], Loss: 117.9163
Epoch [4000/4500], Loss: 118.2764


In [100]:
torch.save(model, "models/model.pth")

In [101]:
model.eval()
with torch.no_grad():
    train_loss = loss(model(X_train_tensor), y_train_tensor).item()
    test_loss = loss(model(X_test_tensor), y_test_tensor).item()
    print(f'Train MSE: {train_loss:.4f}, Test MSE: {test_loss:.4f}')

Train MSE: 114.4321, Test MSE: 112.9972


## Results

| Model | Test MSE | √MSE (≈ avg error, kWh) |
|-------|----------|--------------------------|
| Linear regression | ~121 | ~11 |
| Neural network | ~112 | ~10.9 |

The neural network slightly outperforms the baseline. Further gains may come from longer training, different architectures, or feature selection. **Possible extensions:** different feature sets, hidden sizes, learning rates, or training duration.